<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-01/AI_Modul_1.5_Praktikum_Penerapan_AI.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 1.5: Praktikum Penerapan AI di Berbagai Bidang
### Pipeline Terpadu Multi-Sektor: Smart Agro-Vision (Klasifikasi Kematangan Sawit) & Predictive Maintenance (Deteksi Anomali Mesin Pabrik)

---

> **Diktat Terkait:** [AI_Modul_1.5_Contoh_Penerapan_AI_di_Berbagai_Bidang.md](../../docs/part-01/AI_Modul_1.5_Contoh_Penerapan_AI_di_Berbagai_Bidang.md)  
> **Outputs:** Kode program implementasi multi-sektor industri dengan komentar per baris, evaluasi metrik berbobot risiko ($F_\beta$-score), dan optimasi utilitas multi-objektif (akurasi vs latensi vs biaya).  
> **Outcomes:** Mahasiswa terampil memilih modalitas AI yang tepat untuk masalah industri nyata dan mampu menghitung kelayakan sistem dengan memperhitungkan batasan komputasi *Edge vs Cloud*.  
> **Impacts:** Terwujudnya modernisasi agro-industri kelapa sawit dan otomatisasi pemeliharaan aset pabrik untuk mencegah *unplanned downtime* yang mahal.
>
> **Tujuan Pembelajaran:**  
> 1. Mengimplementasikan model *Computer Vision* berlatensi ultra-rendah untuk pemilahan (*grading*) kematangan buah sawit.  
> 2. Mengimplementasikan model deteksi anomali tanpa label (*Isolation Forest*) untuk memantau sinyal getaran turbin pabrik kelapa sawit.  
> 3. Menghitung dan memvisualisasikan trade-off fungsi utilitas multi-objektif industri.

## 1. Persiapan Environment & Import Library
Jalankan sel berikut untuk memuat pustaka pengolahan numerik, pemodelan machine learning, dan visualisasi grafik:

In [ ]:
# Mengimpor modul sys untuk membaca metadata sistem Python
import sys

# Mengimpor modul os untuk manipulasi path dan sistem berkas
import os

# Mengimpor pustaka numpy untuk operasi vektor dan kalkulasi matriks numerik
import numpy as np

# Mengimpor matplotlib.pyplot untuk merender visualisasi grafik 2D resolusi tinggi
import matplotlib.pyplot as plt

# Mengimpor fungsi pembuat data sintetis klasifikasi dari pustaka scikit-learn
from sklearn.datasets import make_blobs

# Mengimpor algoritma Logistic Regression untuk klasifikasi terawasi berlatensi sangat rendah
from sklearn.linear_model import LogisticRegression

# Mengimpor algoritma Isolation Forest untuk deteksi anomali tanpa label pada sinyal mesin
from sklearn.ensemble import IsolationForest

# Mengimpor metrik evaluasi klasifikasi: confusion matrix dan precision-recall f-score
from sklearn.metrics import confusion_matrix, precision_score, recall_score, fbeta_score

# Menetapkan nilai seed acak agar hasil simulasi numerik selalu konsisten saat diuji ulang
np.random.seed(42)

# Mengatur gaya grafik visualisasi
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120

# Menampilkan versi interpreter Python
print(f"Python Version : {sys.version.split()[0]}")
print("[OK] Seluruh pustaka saintifik berhasil dimuat dengan sukses!")

## 2. Sektor 1: Pertanian Presisi (Agro-Vision Sortasi Buah Sawit)

Pada stasiun penerimaan buah (*loading ramp*) Pabrik Kelapa Sawit (PKS), ribuan ton Tandan Buah Segar (TBS) masuk setiap hari. Memisahkan buah mentah (*unripe*) dari buah matang sempurna (*ripe*) secara manual rentan terhadap kelelahan manusia dan ketidakakuratan subjektif.

Berikut kita melatih model pengklasifikasi linier berlatensi $<1\text{ ms}$ menggunakan dua indeks optik spektral:
* $x_1$: **Red Reflection Index** (karotenoid).
* $x_2$: **Yellowish Spectral Ratio** (kematangan daging buah).

In [ ]:
# Membangkitkan 250 sampel data spektral sintetis buah sawit
# Klaster 0: Mentah ([30, 40]), Klaster 1: Matang ([80, 85])
X_sawit, y_sawit = make_blobs(n_samples=250, centers=[[30, 40], [80, 85]], cluster_std=14, random_state=42)

# Menginisialisasi model Logistic Regression dengan toleransi konvergensi standar
model_sawit = LogisticRegression()

# Melatih model menggunakan data spektral dan label kematangan tandan buah
model_sawit.fit(X_sawit, y_sawit)

# Melakukan prediksi status kematangan pada seluruh sampel
y_pred_sawit = model_sawit.predict(X_sawit)

# Menghitung skor presisi sortasi kematangan buah sawit
prec_sawit = precision_score(y_sawit, y_pred_sawit)

# Menghitung skor recall sortasi kematangan buah sawit
rec_sawit = recall_score(y_sawit, y_pred_sawit)

# Menghitung F1-score (keseimbangan presisi dan recall)
f1_sawit = fbeta_score(y_sawit, y_pred_sawit, beta=1.0)

# Menghitung F0.5-score (presisi diutamakan 2x lebih tinggi untuk meminimalkan buah mentah masuk rebusan)
f05_sawit = fbeta_score(y_sawit, y_pred_sawit, beta=0.5)

print("=" * 60)
print("HASIL EVALUASI SORTASI AGRO-VISION KELAPA SAWIT")
print("=" * 60)
print(f"Total Tandan Buah Diuji : {len(y_sawit)} tandan")
print(f"Precision Kematangan   : {prec_sawit * 100:.2f}% (Kemurnian buah matang)")
print(f"Recall Kematangan      : {rec_sawit * 100:.2f}% (Rasio buah matang terangkut)")
print(f"F1-Score (Seimbang)    : {f1_sawit * 100:.2f}%")
print(f"F0.5-Score (Presisi^)  : {f05_sawit * 100:.2f}%")
print("=" * 60)

# Visualisasi ruang fitur dan batas pemisah AI
fig, ax = plt.subplots(figsize=(8, 5))

# Scatter plot untuk buah mentah
ax.scatter(X_sawit[y_sawit == 0, 0], X_sawit[y_sawit == 0, 1], color='#2ca02c', alpha=0.7, edgecolors='k', label='Buah Mentah (Unripe)')

# Scatter plot untuk buah matang
ax.scatter(X_sawit[y_sawit == 1, 0], X_sawit[y_sawit == 1, 1], color='#ff7f0e', alpha=0.7, edgecolors='k', label='Buah Matang (Ripe)')

# Garis batas keputusan AI: w0*x + w1*y + b = 0
x_vals = np.linspace(X_sawit[:, 0].min() - 5, X_sawit[:, 0].max() + 5, 100)
y_vals = -(model_sawit.coef_[0][0] * x_vals + model_sawit.intercept_[0]) / model_sawit.coef_[0][1]
ax.plot(x_vals, y_vals, 'k--', linewidth=2, label='Batas Keputusan AI')

ax.set_title("Klasifikasi Tingkat Kematangan Buah Kelapa Sawit (Agro-Vision)", fontsize=12, fontweight='bold')
ax.set_xlabel("Red Optical Reflection Index", fontsize=10, fontweight='bold')
ax.set_ylabel("Yellowish Spectral Ratio", fontsize=10, fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 3. Sektor 2: Manufaktur Industri (Predictive Maintenance Turbin PKS)

Turbin uap di pabrik kelapa sawit berputar pada kecepatan ribuan RPM. Kegagalan bantalan peluru (*bearing*) yang tidak terdeteksi dapat mengakibatkan kerusakan fatal bernilai miliaran rupiah.

Karena data kerusakan mesin di pabrik sangat jarang terjadi, kita menggunakan model pembelajaran nir-pengawasan (*Unsupervised Learning*) yaitu **Isolation Forest** yang mengisolasi lonjakan anomali vibrasi tanpa label pelatihan.

In [ ]:
# Membangkitkan 580 jam rekaman getaran normal turbin uap (rata-rata 2.4 mm/s, std 0.35 mm/s)
getaran_normal = np.random.normal(loc=2.4, scale=0.35, size=(580, 1))

# Membangkitkan 20 jam sinyal anomali getaran kritis (rentang 4.0 - 7.2 mm/s)
getaran_anomali = np.random.uniform(low=4.0, high=7.2, size=(20, 1))

# Menggabungkan data normal dan anomali ke dalam satu urutan waktu operasional
X_getaran = np.vstack([getaran_normal, getaran_anomali])

# Label acuan riil (Ground Truth): 1 untuk normal, -1 untuk kondisi anomali bahaya
y_gt = np.array([1] * 580 + [-1] * 20)

# Menginisialisasi algoritma Isolation Forest dengan proporsi kontaminasi 3.3% (20/600)
iso_forest = IsolationForest(contamination=0.033, random_state=42)

# Melatih model deteksi anomali pada sinyal getaran tanpa label
iso_forest.fit(X_getaran)

# Memprediksi kondisi setiap jam operasional mesin (1 = Normal, -1 = Anomali)
y_pred_turbin = iso_forest.predict(X_getaran)

# Menghitung metrik keselamatan operasional pabrik (Safety Recall)
anomali_terdeteksi = np.sum((y_gt == -1) & (y_pred_turbin == -1))
total_anomali = np.sum(y_gt == -1)
safety_recall = anomali_terdeteksi / total_anomali

# Menghitung F2-score (Recall diprioritaskan 2x lebih penting daripada Precision demi keselamatan pabrik)
y_gt_bin = (y_gt == -1).astype(int)
y_pred_bin = (y_pred_turbin == -1).astype(int)
f2_pdm = fbeta_score(y_gt_bin, y_pred_bin, beta=2.0)

print("=" * 60)
print("HASIL DETEKSI ANOMALI PREDICTIVE MAINTENANCE TURBIN PKS")
print("=" * 60)
print(f"Total Durasi Pemantauan : {len(X_getaran)} jam")
print(f"Anomali Kritis Riil    : {total_anomali} kejadian")
print(f"Anomali Terdeteksi AI  : {anomali_terdeteksi} kejadian")
print(f"Safety Recall Mesin    : {safety_recall * 100:.2f}% (Pencegahan kerusakan katastropik)")
print(f"F2-Score (Bobot Safety): {f2_pdm * 100:.2f}%")
print("=" * 60)

# Visualisasi sinyal getaran dan tanda alarm anomali dari AI
fig, ax = plt.subplots(figsize=(10, 5))
jam = np.arange(len(X_getaran))

# Plot sinyal getaran kontinu
ax.plot(jam, X_getaran[:, 0], color='#1f77b4', linewidth=1.2, alpha=0.8, label='Sinyal Getaran Akselerometer (mm/s)')

# Plot titik anomali yang dibunyikan alarm oleh AI
anomali_idx = np.where(y_pred_turbin == -1)[0]
ax.scatter(jam[anomali_idx], X_getaran[anomali_idx, 0], color='crimson', s=55, zorder=5, label='Alarm Anomali AI (Pencegahan Kerusakan)')

# Garis ambang batas standar keamanan industri ISO 10816 (4.5 mm/s)
ax.axhline(y=4.5, color='orange', linestyle='--', linewidth=1.5, label='Batas Toleransi ISO 10816 (4.5 mm/s)')

ax.set_title("Pemantauan Getaran Turbin Pabrik Kelapa Sawit (Predictive Maintenance)", fontsize=12, fontweight='bold')
ax.set_xlabel("Waktu Operasional Mesin (Jam)", fontsize=10, fontweight='bold')
ax.set_ylabel("Kecepatan Getaran (mm/s)", fontsize=10, fontweight='bold')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

## 4. Evaluasi Multi-Objektif: Utilitas Sistem Industri (Edge vs Cloud)

Di industri, sistem AI dievaluasi berdasarkan trade-off antara performa akurasi, latensi waktu respon, dan biaya sewa komputasi server:

$$\mathcal{U}_{\text{Sistem}} = w_{\text{acc}} \cdot \text{Akurasi} - w_{\text{lat}} \cdot \left(\frac{L}{L_{\max}}\right) - w_{\text{cost}} \cdot \left(\frac{C}{C_{\max}}\right)$$

Mari kita simulasikan penentuan model terbaik untuk kasus pemantauan drone perkebunan:

In [ ]:
# Fungsi kalkulasi utilitas multi-objektif sistem AI industri
def hitung_utilitas_sistem(akurasi, latensi, biaya, w_acc=0.5, w_lat=0.3, w_cost=0.2, l_max=100.0, c_max=10.0):
    skor_performa = w_acc * akurasi
    penalti_latensi = w_lat * (latensi / l_max)
    penalti_biaya = w_cost * (biaya / c_max)
    utilitas_bersih = skor_performa - penalti_latensi - penalti_biaya
    return utilitas_bersih

# Parameter Model 1: Edge AI Ringan (MobileNet pada modul Jetson Nano di Drone)
acc_edge, lat_edge, cost_edge = 0.92, 15.0, 0.50
u_edge = hitung_utilitas_sistem(acc_edge, lat_edge, cost_edge)

# Parameter Model 2: Cloud AI Berat (Model Masif di GPU Cloud Server)
acc_cloud, lat_cloud, cost_cloud = 0.98, 90.0, 8.50
u_cloud = hitung_utilitas_sistem(acc_cloud, lat_cloud, cost_cloud)

print("=" * 65)
print("EVALUASI MULTI-OBJEKTIF PEMILIHAN ARSITEKTUR AI")
print("=" * 65)
print(f"Opsi 1: Edge AI Ringan  -> Akurasi: {acc_edge*100:.1f}%, Latensi: {lat_edge} ms, Biaya: ${cost_edge}")
print(f"       -> Skor Utilitas Bersih: {u_edge:.4f}")
print(f"Opsi 2: Cloud AI Berat  -> Akurasi: {acc_cloud*100:.1f}%, Latensi: {lat_cloud} ms, Biaya: ${cost_cloud}")
print(f"       -> Skor Utilitas Bersih: {u_cloud:.4f}")
print("-" * 65)
if u_edge > u_cloud:
    print(">> [REKOMENDASI KEPUTUSAN]: Pilih Opsi 1 (Edge AI) karena lebih efisien & responsif.")
else:
    print(">> [REKOMENDASI KEPUTUSAN]: Pilih Opsi 2 (Cloud AI) karena peningkatan akurasi menjustifikasi biaya.")
print("=" * 65)

## 5. Tugas Mandiri & Eksplorasi Analitis Mahasiswa

1. **Eksplorasi Nilai Beta**: Ubah parameter `beta=0.5` menjadi `beta=3.0` pada evaluasi getaran turbin PKS. Mengapa nilai beta yang tinggi lebih merefleksikan filosofi keselamatan di industri berat?
2. **Sensitivitas Batas Latensi**: Jika sistem drone perkebunan dihubungkan ke jaringan satelit dengan latensi melonjak menjadi $150\text{ ms}$, hitung ulang skor utilitas model Edge AI dan analisis kelayakannya terhadap nilai $L_{\max} = 100\text{ ms}$!